# CF Scrum Pipeline × Tableau Cloud Asset Matcher

This notebook **preserves the original `CF Scrum Pipeline.xlsx`** and enriches the **`Pipeline 2026 Q4`** worksheet with confirmed/potential Tableau asset matches. It uses a PAT to discover accessible projects, workbooks, published data sources, and flows across the Tableau Cloud site.

**Matching is evidence-based:** exact identifiers/name matches are high-confidence candidates, while fuzzy semantic matches are marked **Potential** for review. Text similarity alone is **not proof** that an initiative uses an asset. No Tableau assets are created or modified.

## Step 1 — Install dependencies

Run in the same Jupyter kernel used for your Tableau Cloud work. `openpyxl` preserves the existing Excel workbook structure and formatting, while `rapidfuzz` scores candidate matches.

In [ ]:
%pip install -q requests openpyxl rapidfuzz

## Step 2 — Configuration and imports

Set your source workbook path and Tableau PAT environment variables. The pipeline never stores the PAT secret in the Excel output. Set `inventory_xlsx` to an existing Commercial Finance Inventory Excel if available; its asset descriptions will supplement the live REST discovery.

In [ ]:
import os, re, json, time, getpass, logging, math, copy
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from rapidfuzz import fuzz
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.hyperlink import Hyperlink

CONFIG = {
    "input_xlsx": r"CF Scrum Pipeline.xlsx",
    "output_xlsx": r"CF Scrum Pipeline - Tableau Asset Matches.xlsx",
    "target_sheet": "Pipeline 2026 Q4",
    "tableau_server": "https://your-tableau-pod.online.tableau.com",
    "site_content_url": "demo-site",
    "rest_api_version": "3.29",
    "pat_name_env": "TABLEAU_PAT_NAME",
    "pat_secret_env": "TABLEAU_PAT_SECRET",
    "inventory_xlsx": "",          # optional existing inventory file
    "include_all_projects": True,
    "minimum_potential_score": 35,
    "fuzzy_only_threshold": 75,
    "confirmed_score": 92,         # requires strong lexical evidence; manually verify
    "top_matches_per_initiative": 5,  # legacy setting\n    "top_matches_per_asset_type": 5,  # up to 5 EACH: Workbook, Data Source, Flow\n    "match_asset_types": ("Workbook", "Data Source", "Flow"),
    "timeout_seconds": 60,
    "max_retries": 3,
}
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("tableau_scrum_matcher")

## Step 3 — Read the initiative worksheet

Initiatives are recognized from the existing layout: **A = task ID, B = Deliverable, C = Task, D = FEATURES, E = Applicable Region**. Section headings, subtotals, and blank lines are skipped. Original cells and formulas are not changed.

**Performance fix:** read the target worksheet in streaming mode (`read_only=True`) and use one `iter_rows` pass, instead of opening every sheet for editing during Step 3. The editable workbook is loaded later in Step 8, which can still take time because of the source file’s oversized formatting range.

In [ ]:
source_path = Path(CONFIG["input_xlsx"])
if not source_path.exists():
    raise FileNotFoundError(f"Place the original Excel file beside the notebook or update input_xlsx: {source_path}")
read_workbook = load_workbook(source_path, read_only=True, data_only=True)
if CONFIG["target_sheet"] not in read_workbook.sheetnames:
    raise KeyError(f"Missing worksheet {CONFIG['target_sheet']!r}")
ws = read_workbook[CONFIG["target_sheet"]]
initiatives = []
for row, values in enumerate(ws.iter_rows(min_row=5, max_col=5, values_only=True), start=5):
    task_id = str(values[0] or "").strip()
    title = str(values[1] or "").strip()
    description = str(values[2] or "").strip()
    feature = str(values[3] or "").strip()
    region = str(values[4] or "").strip()
    if not title or not (task_id or description or feature):
        continue
    if not re.match(r"^T\d+$", task_id, flags=re.I):
        continue
    initiatives.append({
        "row":row,"task_id":task_id,"title":title,
        "description":description,"feature":feature,"region":region,
    })
print(f"Found {len(initiatives)} initiative rows in {CONFIG['target_sheet']}.")
print("First initiatives:", [(x["task_id"],x["title"]) for x in initiatives[:4]])
read_workbook.close()


## Step 4 — Authenticate to Tableau Cloud with PAT

The REST API discovers only content accessible to the PAT user. The pipeline paginates results and retries transient HTTP failures. A PAT with broader site visibility produces more complete matching.

In [ ]:
session = requests.Session()
retry = Retry(total=CONFIG["max_retries"], backoff_factor=1,
              status_forcelist=[429,500,502,503,504], allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retry))
BASE = CONFIG["tableau_server"].rstrip("/") + "/api/" + CONFIG["rest_api_version"]
pat_name = os.getenv(CONFIG["pat_name_env"]) or input("Tableau PAT name: ").strip()
pat_secret = os.getenv(CONFIG["pat_secret_env"]) or getpass.getpass("Tableau PAT secret: ")
signin_payload = {"credentials":{"personalAccessTokenName":pat_name,
                                  "personalAccessTokenSecret":pat_secret,
                                  "site":{"contentUrl":CONFIG["site_content_url"]}}}
r = session.post(BASE + "/auth/signin", json=signin_payload, timeout=CONFIG["timeout_seconds"],
                 headers={"Accept":"application/json","Content-Type":"application/json"})
r.raise_for_status()
creds = r.json()["credentials"]
SITE_ID = creds["site"]["id"]
session.headers.update({"X-Tableau-Auth":creds["token"],"Accept":"application/json"})
del pat_secret, signin_payload
print("Authenticated to site:", CONFIG["site_content_url"])

def tableau_get(path, params=None):
    r = session.get(BASE + path, params=params, timeout=CONFIG["timeout_seconds"])
    r.raise_for_status()
    return r.json()

def paged(endpoint, collection, item):
    results=[]; page=1; size=1000
    while True:
        payload=tableau_get(endpoint, {"pageSize":size,"pageNumber":page})
        part=payload.get(collection,{}).get(item,[])
        if isinstance(part,dict): part=[part]
        results.extend(part)
        pagination=payload.get("pagination",{})
        total=int(pagination.get("totalAvailable",len(results)))
        if not part or len(results)>=total: break
        page+=1
    return results

## Step 5 — Discover projects, workbooks, data sources and flows

Build full hierarchical project paths. Collect high-level descriptions and URLs where the REST API exposes them. Inaccessible endpoints are recorded as warnings, not silently treated as empty.

All three asset types are scanned. **Independent per-type matching quotas** ensure that workbook candidates cannot displace potentially relevant flows or published data sources.

In [ ]:
errors=[]
try:
    projects=paged(f"/sites/{SITE_ID}/projects","projects","project")
    workbooks=paged(f"/sites/{SITE_ID}/workbooks","workbooks","workbook")
    datasources=paged(f"/sites/{SITE_ID}/datasources","datasources","datasource")
    try:
        flows=paged(f"/sites/{SITE_ID}/flows","flows","flow")
    except Exception as exc:
        errors.append(f"Flows could not be listed: {exc}")
        flows=[]
    project_by_id={str(p.get("id")):p for p in projects}
    def project_parent(p):
        v=p.get("parentProjectId") or p.get("parentProject") or ""
        return str(v.get("id","")) if isinstance(v,dict) else str(v)
    def project_path(pid):
        parts=[];seen=set()
        while pid and pid not in seen and pid in project_by_id:
            seen.add(pid); p=project_by_id[pid]
            parts.append(str(p.get("name","")))
            pid=project_parent(p)
        return " / ".join(reversed(parts))
    def asset_project_id(a):
        p=a.get("project") or {}
        return str(p.get("id","")) if isinstance(p,dict) else str(p)
    assets=[]
    for kind, records in [("Workbook",workbooks),("Data Source",datasources),("Flow",flows)]:
        for a in records:
            pid=asset_project_id(a)
            assets.append({
                "asset_id":str(a.get("id","")),"name":str(a.get("name","")),
                "type":kind,"project_id":pid,"project_path":project_path(pid),
                "description":str(a.get("description") or ""),
                "content_url":str(a.get("contentUrl") or ""),
                "webpage_url":str(a.get("webpageUrl") or ""),
                "updated_at":str(a.get("updatedAt") or ""),
                "owner_id":str((a.get("owner") or {}).get("id","")) if isinstance(a.get("owner"),dict) else "",
            })
finally:
    try: session.post(BASE+"/auth/signout",timeout=15)
    except Exception: pass
    session.close()
print(f"Projects: {len(projects)} | Workbooks: {len(workbooks)} | Data Sources: {len(datasources)} | Flows: {len(flows)}")
print("Warnings:",errors)

## Step 6 — Optionally enrich assets from an existing inventory

If your Commercial Finance Inventory Explorer output is available, this step merges its descriptions and URLs using Tableau asset IDs. It does **not** substitute an inventory-only scan for site-wide discovery.

In [ ]:
inventory_path=Path(CONFIG["inventory_xlsx"]) if CONFIG["inventory_xlsx"] else None
if inventory_path and inventory_path.exists():
    inv=load_workbook(inventory_path,read_only=True,data_only=True)
    lookup={}
    for tab in ("All Assets","Workbooks","Data Sources","Flows"):
        if tab not in inv.sheetnames: continue
        rows=inv[tab].iter_rows(values_only=True)
        headers=[str(x or "").strip() for x in next(rows,())]
        for values in rows:
            rec=dict(zip(headers,values))
            aid=str(rec.get("Asset ID") or "").strip()
            if aid: lookup[aid]=rec
    for a in assets:
        extra=lookup.get(a["asset_id"],{})
        if not a["description"]: a["description"]=str(extra.get("Description") or "")
        if not a["webpage_url"]: a["webpage_url"]=str(extra.get("Webpage URL") or "")
    inv.close()
    print(f"Enriched using {len(lookup)} inventory asset records.")
else:
    print("No optional inventory supplied; using Tableau REST metadata.")

## Step 7 — Match initiative text to Tableau assets

Scoring considers initiative name, task description, feature description, region, asset name, description, and project path. It uses weighted fuzzy text comparisons plus token overlap. **Confirmed Candidate** means strong textual evidence, not a verified technical dependency; **Potential Match** means review is required. A task may match several assets.

Each initiative receives up to **five candidates per asset type** (15 total): Workbooks, published Data Sources, and Flows. All classifications and matching evidence remain visible.

In [ ]:
# Backward-compatible configuration for existing Jupyter kernels.
# This avoids KeyError when Step 2 was run using an older notebook version.
CONFIG.setdefault("match_asset_types", ("Workbook", "Data Source", "Flow"))
CONFIG.setdefault("top_matches_per_asset_type", 5)

STOPWORDS=set("the and for with from into this that data report dashboard tableau new add create update enhance to of in by on as a an is are be daily weekly monthly logic dataset field source pipeline project task feature".split())
def tokens(s):
    return set(w for w in re.findall(r"[a-z0-9]+", str(s).casefold()) if len(w)>2 and w not in STOPWORDS)
def similarity(initiative, asset):
    title=str(initiative["title"])
    detail=" ".join(str(initiative[k]) for k in ("description","feature","region"))
    name=str(asset["name"])
    context=" ".join(str(asset.get(k) or "") for k in ("description","project_path"))
    left=tokens(title+" "+detail)
    right=tokens(name+" "+context)
    shared=left & right
    name_shared=left & tokens(name)
    exact=title.strip().casefold()==name.strip().casefold()
    phrase=bool(title.strip()) and title.casefold() in (name+" "+context).casefold()
    name_score=fuzz.token_set_ratio(title,name)
    context_score=fuzz.token_set_ratio(title+" "+detail,name+" "+context)
    keyword_score=100*len(shared)/max(1,min(len(left),len(right)))
    score=round(min(100,0.48*name_score+0.27*context_score+0.25*keyword_score),1)
    if exact:
        score=max(score,99);label="Exact Name Match"
    elif phrase or (len(name_shared)>=2 and name_score>=90):
        score=max(score,90);label="Strong Partial Match"
    elif len(shared)>=2:
        score=max(score,65);label="Multi-Keyword Match"
    elif len(shared)==1:
        score=max(score,CONFIG.get("minimum_potential_score",35));label="Single-Keyword Match"
    elif score>=CONFIG.get("fuzzy_only_threshold",75):
        label="Fuzzy Text Match"
    else:
        label="No Match"
    return score,label,", ".join(sorted(shared)[:20])
matches=[]
for ini in initiatives:
    ranked=[]
    for a in assets:
        score,label,shared=similarity(ini,a)
        if label!="No Match":
            ranked.append({**ini,**a,"score":score,"classification":label,"shared_terms":shared})
    priority={"Exact Name Match":0,"Strong Partial Match":1,"Multi-Keyword Match":2,"Single-Keyword Match":3,"Fuzzy Text Match":4}
    ranked.sort(key=lambda x:(priority.get(x["classification"],9),-x["score"],x["name"].lower()))
    # Keep independent candidate quotas: workbook matches cannot crowd out flows/data sources.
    for asset_type in CONFIG["match_asset_types"]:
        matches.extend([x for x in ranked if x["type"] == asset_type]
                       [:CONFIG["top_matches_per_asset_type"]])
print(f"Initiatives loaded: {len(initiatives)} | Tableau assets discovered: {len(assets)}")
print(f"Candidate matches: {len(matches)} for {len(set(x['row'] for x in matches))} initiatives.")
if not initiatives: raise RuntimeError("No initiatives loaded. Check Step 3 and task-ID regex.")
if not assets: print("WARNING: No Tableau assets discovered. Check Step 5.")


## Step 8 — Formula-safe, fast Excel export

This step **copies the original Excel archive** and updates only `Pipeline 2026 Q4` with clearly labelled matching columns. It never replaces existing cells or their formulas, and it adds separate **Tableau Asset Matches** and **Tableau Match Readme** sheets. It verifies that every original formula and its definition remain unchanged before saving.

**Important:** Always set `input_xlsx` to your **original** `CF Scrum Pipeline.xlsx`, not to a previously generated matching output. Previous damaged outputs cannot be used to reconstruct lost formulas.


**Two-way worksheet navigation:** Click the best matched asset or classification in `Pipeline 2026 Q4` to open the first ranked match in `Tableau Asset Matches`. In the match-detail sheet, click an initiative name or source-row link to return to its original row. These are native internal Excel links, not external URLs.

Hyperlinked cells are styled with the conventional **blue underlined font**. Valid Tableau Cloud URLs in the detail worksheet are clickable external hyperlinks.

The source worksheet now includes separate **Workbook Matches**, **Data Source Matches**, and **Flow Matches** counts, while the detail worksheet identifies every candidate's asset type and provides two-way navigation and Tableau Cloud links.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from lxml import etree as ET
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import re, json, copy, posixpath

MAIN = 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'
REL = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'
PKG = 'http://schemas.openxmlformats.org/package/2006/relationships'
CT = 'http://schemas.openxmlformats.org/package/2006/content-types'
NS = {'m': MAIN, 'r': REL}
def q(tag): return '{'+MAIN+'}'+tag

def excel_letter(n):
    result = ''
    while n:
        n, rem = divmod(n-1, 26)
        result = chr(65+rem)+result
    return result

def column_number(ref):
    m = re.match(r'([A-Z]+)', ref or '')
    if not m: return 0
    num = 0
    for letter in m.group(1): num = num*26+ord(letter)-64
    return num

def append_new_cell(row, col, value):
    """Add only NEW cells. Never remove, reformat or overwrite original cells."""
    address = f'{excel_letter(col)}{row.get("r")}'
    if any(c.get('r') == address for c in row.findall(q('c'))):
        raise ValueError(f'Refusing to overwrite original cell {address}')
    cell = ET.Element(q('c'), r=address)
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        ET.SubElement(cell, q('v')).text = str(value)
    else:
        cell.set('t', 'inlineStr')
        is_node = ET.SubElement(cell, q('is'))
        ET.SubElement(is_node, q('t')).text = str(value if value is not None else '')
    # Keep existing cell elements and their order untouched; new cells are appended to the right.
    row.append(cell)
    return cell

def simple_sheet_xml(rows, widths):
    root = ET.Element(q('worksheet'), nsmap={None: MAIN})
    views = ET.SubElement(root, q('sheetViews'))
    view = ET.SubElement(views, q('sheetView'), workbookViewId='0')
    ET.SubElement(view, q('pane'), ySplit='1', topLeftCell='A2', activePane='bottomLeft', state='frozen')
    cols = ET.SubElement(root, q('cols'))
    for index, width in enumerate(widths, 1):
        ET.SubElement(cols, q('col'), min=str(index), max=str(index), width=str(width), customWidth='1')
    data = ET.SubElement(root, q('sheetData'))
    for number, values in enumerate(rows, 1):
        row = ET.SubElement(data, q('row'), r=str(number))
        for index, value in enumerate(values, 1): append_new_cell(row, index, value)
    ET.SubElement(root, q('autoFilter'), ref=f'A1:{excel_letter(max(map(len, rows), default=1))}{max(1,len(rows))}')
    return ET.tostring(root, encoding='UTF-8', xml_declaration=True)

def add_internal_links(root, entries):
    """Insert native Excel internal hyperlinks without changing existing cells/formulas."""
    if not entries:
        return
    links = root.find(q('hyperlinks'))
    if links is None:
        links = ET.Element(q('hyperlinks'))
        # ECMA-376 order: hyperlinks follow dataValidations and precede printOptions/pageMargins.
        before = {'printOptions','pageMargins','pageSetup','headerFooter',
                  'rowBreaks','colBreaks','customProperties','cellWatches',
                  'ignoredErrors','smartTags','drawing','legacyDrawing',
                  'legacyDrawingHF','picture','oleObjects','controls','webPublishItems',
                  'tableParts','extLst'}
        index = next((i for i, child in enumerate(root)
                      if ET.QName(child).localname in before), len(root))
        root.insert(index, links)
    existing = {h.get('ref') for h in links.findall(q('hyperlink'))}
    for cell_ref, location, label in entries:
        if cell_ref in existing:
            raise ValueError(f'Hyperlink already exists at {cell_ref}')
        ET.SubElement(links, q('hyperlink'), ref=cell_ref,
                      location=location, display=str(label),
                      tooltip='Navigate to the related worksheet')
        existing.add(cell_ref)

def create_hyperlink_style(styles_root):
    """Append Excel's conventional blue-underlined font style; preserve every existing style."""
    fonts = styles_root.find(q('fonts'))
    xfs = styles_root.find(q('cellXfs'))
    if fonts is None or xfs is None:
        raise ValueError('Workbook styles.xml has no fonts/cellXfs')
    font = ET.Element(q('font'))
    ET.SubElement(font, q('sz'), val='11')
    ET.SubElement(font, q('color'), rgb='FF0563C1')
    ET.SubElement(font, q('name'), val='Calibri')
    ET.SubElement(font, q('u'))
    font_id = len(fonts)
    fonts.append(font)
    fonts.set('count', str(len(fonts)))
    xf = copy.deepcopy(xfs[0])
    xf.set('fontId', str(font_id))
    xf.set('applyFont', '1')
    style_id = len(xfs)
    xfs.append(xf)
    xfs.set('count', str(len(xfs)))
    return style_id

def apply_link_style(root, refs, style_id):
    wanted = set(refs)
    for cell in root.xpath('./m:sheetData/m:row/m:c', namespaces=NS):
        if cell.get('r') in wanted:
            cell.set('s', str(style_id))
            wanted.remove(cell.get('r'))
    if wanted:
        raise ValueError(f'Cannot style missing hyperlink cells: {sorted(wanted)[:10]}')

def add_external_links(root, entries, rels_root):
    """Add clickable HTTP(S) links with correct worksheet relationship IDs."""
    links = root.find(q('hyperlinks'))
    if links is None:
        add_internal_links(root, [])
        links = ET.Element(q('hyperlinks'))
        before = {'printOptions','pageMargins','pageSetup','headerFooter','rowBreaks',
                  'colBreaks','customProperties','cellWatches','ignoredErrors','drawing',
                  'legacyDrawing','tableParts','extLst'}
        index = next((i for i, ch in enumerate(root)
                      if ET.QName(ch).localname in before), len(root))
        root.insert(index, links)
    existing = {h.get('ref') for h in links.findall(q('hyperlink'))}
    used = {r.get('Id') for r in rels_root}
    next_id = 1
    for cell_ref, url in entries:
        if cell_ref in existing:
            raise ValueError(f'Duplicate hyperlink cell {cell_ref}')
        if not str(url).lower().startswith(('https://','http://')):
            continue
        while f'rId{next_id}' in used:
            next_id += 1
        rid = f'rId{next_id}'
        used.add(rid)
        ET.SubElement(rels_root, '{'+PKG+'}Relationship',
                      Id=rid, Type=REL+'/hyperlink', Target=str(url), TargetMode='External')
        ET.SubElement(links, q('hyperlink'), ref=cell_ref,
                      attrib={'{'+REL+'}id':rid}, tooltip='Open Tableau Cloud in your browser')
        existing.add(cell_ref)

def find_sheet_path(zf, workbook_xml, workbook_rels, sheet_name):
    sheets = workbook_xml.find(q('sheets'))
    sheet = next((s for s in sheets if s.get('name') == sheet_name), None)
    if sheet is None: raise KeyError(f'Sheet not found: {sheet_name}')
    rel_id = sheet.get('{'+REL+'}id')
    rel = next(r for r in workbook_rels if r.get('Id') == rel_id)
    target = rel.get('Target')
    path = target.lstrip('/') if target.startswith('/') else posixpath.normpath(posixpath.join('xl', target))
    return path, sheets

def formula_snapshot(root):
    return {cell.get('r'): (ET.tostring(cell.find(q('f'))) if cell.find(q('f')) is not None else None)
            for cell in root.xpath('./m:sheetData/m:row/m:c[m:f]', namespaces=NS)}

by_row = defaultdict(list)
for match in matches: by_row[match['row']].append(match)
summary_headers = ['Tableau Match Classification', 'Best Matched Tableau Asset',
                   'Matched Asset Type', 'Match Score (0-100)', 'Matched Asset Project Path',
                   'Matched Tableau Asset LUID', 'Matching Keywords / Evidence',
                   'Number of Candidate Assets']
detail_headers = ['Task ID', 'Initiative Row', 'Initiative Name', 'Initiative Description',
                  'Match Classification', 'Match Score', 'Tableau Asset Name', 'Asset Type',
                  'Tableau Project Path', 'Tableau Asset LUID', 'Asset Description',
                  'Matched Keywords', 'Asset Last Modified', 'Tableau Cloud URL', 'Source Initiative Row']
detail_rows = [detail_headers]
first_detail_row = {}
for item in matches:
    first_detail_row.setdefault(item['row'], len(detail_rows) + 1)
    detail_rows.append([item['task_id'], item['row'], item['title'], item['description'],
        item['classification'], item['score'], item['name'], item['type'], item['project_path'],
        item['asset_id'], item['description'], item['shared_terms'], item['updated_at'],
        item['webpage_url'], item['row']])
notes_rows = [
    ['Item','Description'],
    ['Run UTC',datetime.now(timezone.utc).isoformat()],
    ['Initiatives scanned',len(initiatives)],
    ['Tableau assets scanned',len(assets)],
    ['Candidate matches',len(matches)],
    ['Exact Name Match','Initiative and asset names equal after case-insensitive normalization'],
    ['Strong Partial Match','Strong phrase/name overlap; human review required'],
    ['Multi-Keyword Match','Two or more meaningful shared keywords; human review required'],
    ['Single-Keyword Match','One meaningful shared keyword; weak evidence'],
    ['Fuzzy Text Match','High fuzzy similarity; human review required'],
    ['No Match / Review','No qualifying candidate found'],
    ['Important','Matches are textual suggestions, not verified Tableau technical dependencies'],
    ['Formula integrity','Original source worksheet formula cells preserved without modification'],
    ['Warnings','; '.join(errors) or 'None'],
]
source_path = Path(CONFIG['input_xlsx'])
output_path = Path(CONFIG['output_xlsx'])
if source_path.resolve() == output_path.resolve():
    raise ValueError('Output must be a different file; original Excel will not be overwritten')
output_path.parent.mkdir(parents=True, exist_ok=True)

with ZipFile(source_path, 'r') as original:
    book = ET.fromstring(original.read('xl/workbook.xml'))
    rels = ET.fromstring(original.read('xl/_rels/workbook.xml.rels'))
    types = ET.fromstring(original.read('[Content_Types].xml'))
    target_path, sheets = find_sheet_path(original, book, rels, CONFIG['target_sheet'])
    root = ET.fromstring(original.read(target_path))
    styles_root = ET.fromstring(original.read('xl/styles.xml'))
    link_style_id = create_hyperlink_style(styles_root)
    original_formulas = formula_snapshot(root)
    data = root.find(q('sheetData'))
    existing_rows = {int(r.get('r')):r for r in data.findall(q('row'))}
    existing_max_col = max((column_number(c.get('r')) for c in root.xpath('./m:sheetData/m:row/m:c', namespaces=NS)), default=0)
    first_col = existing_max_col + 2
    if first_col + len(summary_headers) - 1 > 16384:
        raise ValueError('Not enough free Excel columns for match summary')
    header_row = existing_rows.get(4)
    if header_row is None:
        header_row = ET.Element(q('row'), r='4')
        existing_rows[4] = header_row
        data.append(header_row)
    for offset, title in enumerate(summary_headers):
        append_new_cell(header_row, first_col+offset, title)
    summary_links = []
    for initiative in initiatives:
        row_number = initiative['row']
        items = by_row[row_number]
        best = items[0] if items else None
        values = [best['classification'] if best else 'No Match / Review',
                  best['name'] if best else '', best['type'] if best else '',
                  best['score'] if best else '', best['project_path'] if best else '',
                  best['asset_id'] if best else '', best['shared_terms'] if best else '',len(items)]
        row = existing_rows.get(row_number)
        if row is None:
            row = ET.Element(q('row'), r=str(row_number))
            data.append(row)
            existing_rows[row_number] = row
        for offset, value in enumerate(values): append_new_cell(row, first_col+offset, value)
        if best is not None:
            dest = f"'Tableau Asset Matches'!A{first_detail_row[row_number]}"
            summary_links.append((f'{excel_letter(first_col+1)}{row_number}', dest,
                                  'Open ranked Tableau asset matches'))
            summary_links.append((f'{excel_letter(first_col)}{row_number}', dest,
                                  'Open match evidence'))
    # Keep row ordering valid without reordering existing cells or non-cell elements.
    data[:] = sorted(list(data), key=lambda element: int(element.get('r', '0')))
    dim = root.find(q('dimension'))
    if dim is not None:
        dim.set('ref', f'A1:{excel_letter(first_col+len(summary_headers)-1)}{max(existing_rows)}')
    add_internal_links(root, summary_links)
    apply_link_style(root, [ref for ref, _, _ in summary_links], link_style_id)
    # Do not rewrite existing column definitions, styles, merged cells, validations, or formulas.
    patched = ET.tostring(root, encoding='UTF-8', xml_declaration=True)
    check = ET.fromstring(patched)
    assert formula_snapshot(check) == original_formulas, 'Original formulas changed: refusing to save'

    existing_numbers = [int(m.group(1)) for path in original.namelist()
                        if (m := re.fullmatch(r'xl/worksheets/sheet(\d+)\.xml', path))]
    next_number = max(existing_numbers, default=0) + 1
    existing_rel_ids = [int(m.group(1)) for rel in rels if (m := re.fullmatch(r'rId(\d+)', rel.get('Id','')))]
    next_rel_id = max(existing_rel_ids, default=0) + 1
    next_sheet_id = max(int(s.get('sheetId')) for s in sheets) + 1
    new_parts = {}
    for title, rows, widths in [
        ('Tableau Asset Matches', detail_rows, [18,18,40,55,25,15,43,19,50,40,55,35,25,60,22]),
        ('Tableau Match Readme', notes_rows, [36,110]),
    ]:
        if any(s.get('name') == title for s in sheets):
            raise ValueError(f'Output sheet {title!r} already exists. Use the original input Excel.')
        part = f'xl/worksheets/sheet{next_number}.xml'
        sheet_xml = simple_sheet_xml(rows, widths)
        if title == 'Tableau Asset Matches':
            detail_root = ET.fromstring(sheet_xml)
            back_links = []
            for detail_index, item in enumerate(matches, 2):
                destination = f"'{CONFIG['target_sheet']}'!B{item['row']}"
                back_links.append((f'C{detail_index}', destination, 'Return to initiative'))
                back_links.append((f'O{detail_index}', destination, 'Go to source initiative'))
            add_internal_links(detail_root, back_links)
            external_links = [(f'N{idx}', item['webpage_url'])
                              for idx, item in enumerate(matches, 2)
                              if str(item.get('webpage_url') or '').startswith(('https://','http://'))]
            detail_rels = ET.Element('{'+PKG+'}Relationships', nsmap={None: PKG})
            add_external_links(detail_root, external_links, detail_rels)
            apply_link_style(detail_root,
                             [ref for ref, _, _ in back_links] + [ref for ref, _ in external_links],
                             link_style_id)
            if external_links:
                rel_path = f'xl/worksheets/_rels/sheet{next_number}.xml.rels'
                new_parts[rel_path] = ET.tostring(detail_rels, encoding='UTF-8', xml_declaration=True)
            sheet_xml = ET.tostring(detail_root, encoding='UTF-8', xml_declaration=True)
        new_parts[part] = sheet_xml
        ET.SubElement(sheets, q('sheet'), name=title, sheetId=str(next_sheet_id),
                      attrib={'{'+REL+'}id':f'rId{next_rel_id}'})
        ET.SubElement(rels, '{'+PKG+'}Relationship', Id=f'rId{next_rel_id}',
                      Type=REL+'/worksheet', Target=f'worksheets/sheet{next_number}.xml')
        ET.SubElement(types, '{'+CT+'}Override', PartName='/'+part,
                      ContentType='application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml')
        next_number += 1; next_rel_id += 1; next_sheet_id += 1
    replacements = {'xl/styles.xml':ET.tostring(styles_root, encoding='UTF-8', xml_declaration=True),
                    target_path:patched,
                    'xl/workbook.xml':ET.tostring(book, encoding='UTF-8', xml_declaration=True),
                    'xl/_rels/workbook.xml.rels':ET.tostring(rels, encoding='UTF-8', xml_declaration=True),
                    '[Content_Types].xml':ET.tostring(types, encoding='UTF-8', xml_declaration=True),
                    **new_parts}
    with ZipFile(output_path, 'w') as result:
        for member in original.infolist():
            result.writestr(member, replacements.pop(member.filename, original.read(member.filename)))
        for name, content in replacements.items():
            result.writestr(name, content, compress_type=ZIP_DEFLATED)

with ZipFile(output_path) as check_zip:
    assert check_zip.testzip() is None, 'Output XLSX archive failed integrity check'
    check_root = ET.fromstring(check_zip.read(target_path))
    assert formula_snapshot(check_root) == original_formulas, 'Output workbook formulas differ from source'
    summary_link_count = len(check_root.findall('.//'+q('hyperlink')))
    details_root = ET.fromstring(check_zip.read(next(x for x in new_parts if x.endswith(f'sheet{next_number-2}.xml'))))
    detail_link_count = len(details_root.findall('.//'+q('hyperlink')))
    assert summary_link_count >= len(summary_links), 'Summary navigation links missing'
    expected_external = sum(bool(str(x.get('webpage_url') or '').startswith(('https://','http://'))) for x in matches)
    assert detail_link_count == 2*len(matches) + expected_external, 'Detail navigation/URL links missing'
    print(f'Hyperlinks: {len(summary_links)} forward, {2*len(matches)} return, {expected_external} Tableau Cloud URLs')
print(f'Saved formula-safe matched workbook: {output_path.resolve()}')
print(f'Original formulas preserved: {len(original_formulas)}')
print(f'Match summary columns: {excel_letter(first_col)} to {excel_letter(first_col+len(summary_headers)-1)}')
print(f'Initiatives: {len(initiatives)}; candidate matches: {len(matches)}')


## Step 9 — Review the matched workbook

Open the newly generated file (not the original). On **Pipeline 2026 Q4**, the original formulas remain in place; the new match fields are appended to the right of the existing columns. Review **Tableau Asset Matches** for every candidate and **Tableau Match Readme** for the evidence definitions. If Excel asks to recalculate links to the original `2026 task list`, keep the workbook together; original VLOOKUP formulas are retained.

If you need to rerun the matching, start from the original `CF Scrum Pipeline.xlsx` rather than an earlier enriched copy.


**Navigation:** Click the **Best Matched Tableau Asset** or **Tableau Match Classification** to jump to the corresponding detail row. Click the **Initiative Name** or **Source Initiative Row** in the detail sheet to return. Original formulas remain untouched.